# Dictionary Baseline 개선 실험

기존 Baseline의 한계를 개선하기 위해

1. 특수문자 제거
2. 반복문자 정규화
3. 초성 및 유사발음 사전 추가
4. 빈도 기반 Dictionary 구축

을 수행하였다.

### 라이브러리 및 데이터 불러오기

In [3]:
import pandas as pd
import re

from collections import defaultdict, Counter
from sklearn.model_selection import train_test_split

In [4]:
train = pd.read_csv("/content/train.csv", encoding="utf-8-sig")
test = pd.read_csv("/content/test.csv", encoding="utf-8-sig")

print(train.shape)
print(test.shape)

train.head()

(11263, 3)
(1689, 2)


,ID,input,output
0,TRAIN_00000,별 한 게토 았깝땀. 왜 싸람듯릭 펼 1캐를 쥰눈징 컥꺾폰 싸람믐롯섞 맒록 섧멍핥쟈...,별 한 개도 아깝다. 왜 사람들이 별 1개를 주는지 겪어본 사람으로서 말로 설명하자...
1,TRAIN_00001,잚많 쟉꼬 갉 태 좋눼욥. 차못동 줆 ㅋ,잠만 자고 갈 때 좋네요. 잠옷도 줌 ㅋ
2,TRAIN_00002,절테 간면 않 된는 굣 멥몫,절대 가면 안 되는 곳 메모
3,TRAIN_00003,야... 칵컥 좋꾜 부됴 뼝 뚫렷썹 신원햐쥠만 닮패 넴센 밌쪄벅림. 샥퀘 핥류만 묵...,아... 가격 좋고 뷰도 뻥 뚫려서 시원하지만 담배 냄새 미쳐버림. 싸게 하루만 묵...
4,TRAIN_00004,집윈 축쳐눌료 딴너왓눈뎁 카셩뷔 좋곱 칼쿰한네올. 쩌럼한뒈 뮬콰 욺료토 잊쿄 빻토 ...,지인 추천으로 다녀왔는데 가성비 좋고 깔끔하네요. 저렴한데 물과 음료도 있고 방도 ...


### Train / Validation 분리

In [5]:
train_data, val_data = train_test_split(
    train,
    test_size=0.2,
    random_state=42
)

### 텍스트 전처리

In [6]:
def preprocess_text(text):

    text = str(text)

    text = re.sub(
        r"[^가-힣0-9a-zA-Z\sㄱ-ㅎㅏ-ㅣ~!]",
        "",
        text
    )

    text = re.sub(
        r"(ㅋ|ㅎ){3,}",
        r"\1\1",
        text
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    return text

### 빈도 기반 Dictionary 생성

In [7]:
word_map = defaultdict(list)

for input_text, output_text in zip(
    train_data["input"],
    train_data["output"]
):

    input_words = preprocess_text(input_text).split()
    output_words = str(output_text).split()

    for iw, ow in zip(input_words, output_words):
        word_map[iw].append(ow)

match_dict = {}

for word, outputs in word_map.items():

    most_common_output = (
        Counter(outputs)
        .most_common(1)[0][0]
    )

    match_dict[word] = most_common_output

print("Dictionary Size :", len(match_dict))

Dictionary Size : 138123


### 사용자 정의 Dictionary 추가
- 초성 표현, 자주 등장하는 유사 발음을 Dictionary에 추가

In [14]:
extra_dict = {

    "ㅊㄱ":"최고",
    "ㄱㅅ":"감사",
    "ㅇㅈ":"인정",
    "ㅂㄹ":"별로",
    "ㅈㅁ":"정말",
    "ㄹㅇ":"리얼",

    "ㅁㅊ":"미쳤다",
    "ㅅㅌㅊ":"좋다",
    "ㅍㅌㅊ":"보통이다",
    "ㅎㅌㅊ":"별로다",

    "조타":"좋다",
    "마싯다":"맛있다",
    "머찌다":"멋지다",

    "짱맛":"정말 맛있다",
    "개꿀":"정말 좋다",
    "꿀맛":"맛있다",
    "존맛":"정말 맛있다",
    "존맛탱":"정말 맛있다",

    "갓성비":"가성비가 좋다",
    "개추천":"추천한다",

    "굿굿":"좋다",
    "굿":"좋다",

    "쏘쏘":"보통이다",
    "노맛":"맛없다",

    "존좋":"정말 좋다",
    "존나":"정말",
    "개좋":"정말 좋다",

    "개맛":"정말 맛있다",
    "존맛탱구리":"정말 맛있다",

    "짱":"좋다",

    "꿀잠":"잠을 잘 잤다",

    "꿀잼":"재미있다",
    "노잼":"재미없다",

    "핵맛":"정말 맛있다",
    "핵노잼":"정말 재미없다",

    "강추":"추천한다",
    "비추":"추천하지 않는다",

    "대박":"좋다",
    "굳":"좋다",

    "ㅅㅅ":"좋다",
    "ㄳ":"감사",

    "ㄴㄴ":"아니다",
    "ㅇㅇ":"응",
    "ㅇㅋ":"좋다",

    "ㄱㄱ":"가자",
    "ㄷㄷ":"놀랍다"
}

match_dict.update(extra_dict)

### 문장 단위 Dictionary 구축


In [15]:
sentence_dict = {}

for inp, out in zip(
    train_data["input"],
    train_data["output"]
):

    sentence_dict[inp] = out

    sentence_dict[preprocess_text(inp)] = out

    sentence_dict[inp.replace(" ", "")] = out

    sentence_dict[
        preprocess_text(inp).replace(" ", "")
    ] = out

print("Sentence Dictionary :", len(sentence_dict))

Sentence Dictionary : 34237


### 리뷰 복원 함수

In [17]:
def replace_words(input_text):

    # 문장 전체 매칭
    if input_text in sentence_dict:
        return sentence_dict[input_text]

    clean_input = preprocess_text(input_text)

    if clean_input in sentence_dict:
        return sentence_dict[clean_input]

    # 공백 제거 버전도 확인
    no_space_input = clean_input.replace(" ", "")

    if no_space_input in sentence_dict:
        return sentence_dict[no_space_input]

    # 단어 매칭
    words = clean_input.split()

    replaced_words = []

    for word in words:

        # 정확히 일치하는 경우
        if word in match_dict:
            replaced_words.append(match_dict[word])

        else:

            # 부분 문자열 매칭
            new_word = word

            for key in match_dict:

                if len(key) >= 2 and key in word:

                    new_word = new_word.replace(
                        key,
                        match_dict[key]
                    )

            replaced_words.append(new_word)

    return " ".join(replaced_words)

### Validation 데이터 복원
- 검증 데이터와 난독화 문장 복원

In [18]:
val_pred = val_data["input"].apply(
    replace_words
)

val_pred.head()

,input
6927,오션퓨 가성비 숙소웩오! 별섶 두 펀쳅 빵물닒예옷! 캠핑 왔는데 완전 힐링하고......
1669,바다예셔 갖캅귄 한데 수 태뮨엔 전혀 바다갔 뽀이쥐 않는 굣윕님타 칵깝임 있어됴 전...
9499,"찜꿀루웨 떡폭뀌위치 낌찧칟개인짇 국물이 묻뗘있고, 침구루 청소는 않옜 안 하니 봐요..."
3287,"주변예 룟뒈빼꽈쩜밑 있고 음식 싸왈섦 편히 잘 놀다 왓엶오 좋은 호텔른 아니,곬 그..."
2973,"쀼룝 할 이 단, 함!! 룸 컨디션 최상.!! 굴렇냐 옐뤼뻬인떡갔 약간 훈들려 풀않..."


### Validation Accuracy 측정

In [19]:
accuracy = (
    val_pred ==
    val_data["output"]
).mean()

print("Validation Accuracy :", accuracy)

Validation Accuracy : 0.0031069684864624943


### Character F1 Score 계산

In [20]:
def char_f1(pred, true):

    matches = sum(
        1
        for a, b in zip(pred, true)
        if a == b
    )

    precision = matches / max(len(pred), 1)
    recall = matches / max(len(true), 1)

    if precision + recall == 0:
        return 0

    return (
        2 * precision * recall
        / (precision + recall)
    )

scores = []

for pred, true in zip(
    val_pred,
    val_data["output"]
):

    scores.append(
        char_f1(pred, true)
    )

print(
    "Validation Char F1 :",
    sum(scores) / len(scores)
)

Validation Char F1 : 0.31785708126885204


### Test 데이터 복원

- 구축한 Dictionary를 이용하여 test 데이터 전체를 복원

- 복원 결과는 제출 파일 생성에 사용됨

In [21]:
converted_reviews = test["input"].apply(
    replace_words
).tolist()

#### 제출 파일 생성

In [22]:
submission = pd.read_csv(
    "/content/sample_submission.csv",
    encoding="utf-8-sig"
)

submission["output"] = converted_reviews

submission.to_csv(
    "submission_dictionary_improved.csv",
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료")

저장 완료
